# EDA — Hanoi Hourly→Daily Weather Dataset (Auto‑target)

**Dataset:** `hourly_to_daily_weather.csv`

In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = r'data/hourly_to_daily_weather.csv'

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

def rolling_corr(a, b, window=90):
    return a.rolling(window=window, min_periods=max(10, window//3)).corr(b)

def plot_line(df, x, y, title, ylabel=None, rolling=None):
    if y not in df.columns: 
        print(f"[skip] {y} not in columns"); 
        return
    plt.figure(figsize=(12,4))
    s = df[y]
    if rolling is not None:
        s = s.rolling(rolling, min_periods=max(2, rolling//4)).mean()
    plt.plot(df[x], s)
    plt.title(title)
    plt.xlabel(x); plt.ylabel(y if ylabel is None else ylabel)
    plt.tight_layout(); plt.show()

def plot_dual_axis(df, x, y_left, y_right, title, left_label=None, right_label=None, rolling=30):
    if y_left not in df.columns or y_right not in df.columns:
        print(f"[skip] dual-axis ({y_left}, {y_right}) not available")
        return
    plt.figure(figsize=(12,4))
    ax1 = plt.gca()
    s_left = df[y_left].rolling(rolling, min_periods=max(2, rolling//4)).mean()
    ax1.plot(df[x], s_left)
    ax1.set_xlabel(x); ax1.set_ylabel(left_label or y_left)
    ax2 = ax1.twinx()
    s_right = df[y_right].rolling(rolling, min_periods=max(2, rolling//4)).mean()
    ax2.plot(df[x], s_right)
    ax2.set_ylabel(right_label or y_right)
    plt.title(title)
    plt.tight_layout(); plt.show()

def plot_scatter(df, x, y, title, sample=5000):
    if x not in df.columns or y not in df.columns:
        print(f"[skip] scatter ({x},{y}) not available")
        return
    plt.figure(figsize=(6,6))
    d = df[[x,y]].dropna()
    if len(d) > sample: d = d.sample(sample, random_state=42)
    plt.scatter(d[x], d[y], s=8, alpha=0.6)
    plt.title(title); plt.xlabel(x); plt.ylabel(y)
    plt.tight_layout(); plt.show()

def plot_box_by_quantiles(df, feature, target, q=4, title=''):
    if feature not in df.columns or target not in df.columns:
        print(f"[skip] box by quantiles not available for {feature}/{target}")
        return
    vals = df[[feature, target]].dropna()
    qs = np.linspace(0, 1, q+1)
    bins = np.unique(vals[feature].quantile(qs).values)
    if len(bins) < 3:
        print(f'[skip] not enough variation to bin {feature}')
        return
    labels = [f'Q{i+1}' for i in range(len(bins)-1)]
    groups = pd.cut(vals[feature], bins=bins, include_lowest=True, labels=labels)
    to_plot = [vals[target][groups==lab] for lab in labels]
    plt.figure(figsize=(8,5))
    plt.boxplot(to_plot, showfliers=False)
    plt.xticks(range(1, len(labels)+1), labels)
    plt.title(title if title else f'{target} by {feature} quantiles')
    plt.ylabel(target)
    plt.tight_layout(); plt.show()

def plot_heatmap(matrix, xticks, yticks, title):
    plt.figure(figsize=(10,6))
    plt.imshow(matrix, aspect='auto', interpolation='nearest')
    plt.xticks(range(len(xticks)), xticks, rotation=90)
    plt.yticks(range(len(yticks)), yticks)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()


## 1) Load & Overview

In [4]:

df = pd.read_csv(CSV_PATH)
# detect datetime-like column
date_col = None
for c in df.columns:
    if c.lower() in ('datetime','date','day','ds','time'):
        date_col = c; break
if date_col is None:
    # try to infer
    for c in df.columns:
        try:
            pd.to_datetime(df[c].iloc[0]); date_col = c; break
        except Exception: pass
if date_col is None:
    raise ValueError("No datetime-like column found. Please ensure a date/datetime column exists.")

df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df = df.sort_values(date_col).reset_index(drop=True)

print('Shape:', df.shape)
print('Date range:', df[date_col].min(), '→', df[date_col].max())
print('Columns:', len(df.columns))
df.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'data/hourly_to_daily_weather.csv'

## 2) Info & Missingness

In [ ]:

non_null = df.notna().sum()
nulls = df.isna().sum()
dtypes = df.dtypes.astype(str)
overview = pd.DataFrame({'dtype': dtypes, 'non_null': non_null, 'nulls': nulls, 'null_pct': (nulls/len(df))*100})
overview.sort_values(['null_pct','dtype'], ascending=[False, True]).head(30)


In [ ]:

numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
df[numeric_cols].describe().T.head(20)


## 3) Target Detection (No‑Creation Policy)

In [ ]:

# Only use an existing column as target; do not create any.
cands_primary = [c for c in df.columns if c.startswith(('temp_daily_mean_t+','temp_next_','temp_daily_max_t+','temp_daily_min_t+'))]
cands_fallback = [c for c in df.columns if 'temp' in c.lower()]  # e.g., 'temp', 'temp_daily_mean', etc.

def rank(cols):
    def score(name):
        n = name.lower()
        s = 0
        if 't+' in n or 'next_' in n: s += 100
        if 'daily' in n: s += 50
        if 'mean' in n: s += 20
        if n == 'temp': s += 5
        return -s
    return sorted(cols, key=score)

target_col = None
if cands_primary:
    target_col = rank(cands_primary)[0]
elif cands_fallback:
    target_col = rank(cands_fallback)[0]

print("Target column detected:", target_col)


## 4) Group Columns (auto-detect)

In [ ]:

def pick_first(*cands):
    for c in cands:
        if c in df.columns: return c
    return None

groups = {
    'temperature_mean': pick_first('temp_win24h_mean','temp_win12h_mean','temp_mean','temp'),
    'temperature_std' : pick_first('temp_win48h_std','temp_win24h_std'),
    'dew_spread'      : pick_first('dew_spread_win24h_mean','dew_spread'),
    'humidity'        : pick_first('humidity_win24h_mean','humidity'),
    'precip'          : pick_first('precip_win24h_sum','precip'),
    'cloudcover'      : pick_first('cloudcover_win24h_mean','cloudcover'),
    'visibility'      : pick_first('visibility_win24h_mean','visibility'),
    'windspeed'       : pick_first('windspeed_win24h_mean','windspeed'),
    'windgust'        : pick_first('windgust_win12h_max','windgust'),
    'solarradiation'  : pick_first('solarradiation_win24h_mean','solarradiation'),
    'uvindex'         : pick_first('uvindex_win24h_mean','uvindex'),
    'sealevelpressure': pick_first('sealevelpressure_win24h_mean','sealevelpressure'),
}
groups


## 5) Visualizations by Group

### 5.1 Temperature

In [ ]:

x = date_col
if groups['temperature_mean'] is not None and target_col is not None:
    plot_dual_axis(df, x, groups['temperature_mean'], target_col,
                   title=f"{groups['temperature_mean']} vs {target_col}", rolling=30)
    plot_scatter(df, groups['temperature_mean'], target_col,
                 title=f"Scatter: {groups['temperature_mean']} vs {target_col}")
elif groups['temperature_mean'] is not None:
    plot_line(df, x, groups['temperature_mean'], title=f"{groups['temperature_mean']} over time", rolling=30)

if groups['temperature_std'] is not None:
    plot_line(df, x, groups['temperature_std'], title=f"{groups['temperature_std']} over time", rolling=30)
    if target_col is not None:
        rc = rolling_corr(df[groups['temperature_std']], df[target_col], window=90)
        plt.figure(figsize=(12,3)); plt.plot(df[x], rc)
        plt.title(f"Rolling (90d) Corr: {groups['temperature_std']} vs {target_col}")
        plt.xlabel(x); plt.ylabel("Pearson r"); plt.tight_layout(); plt.show()

if groups['dew_spread'] is not None:
    if target_col is not None:
        plot_dual_axis(df, x, groups['dew_spread'], target_col,
                       title=f"{groups['dew_spread']} vs {target_col}", rolling=30)
    else:
        plot_line(df, x, groups['dew_spread'], title=f"{groups['dew_spread']} over time", rolling=30)


### 5.2 Humidity & Dew

In [ ]:

if groups['humidity'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['humidity'], target_col,
                   title=f"{groups['humidity']} vs {target_col}", rolling=30)
    plot_scatter(df, groups['humidity'], target_col,
                 title=f"Scatter: {groups['humidity']} vs {target_col}")
    rc = rolling_corr(df[groups['humidity']], df[target_col], window=90)
    plt.figure(figsize=(12,3)); plt.plot(df[date_col], rc)
    plt.title(f"Rolling (90d) Corr: {groups['humidity']} vs {target_col}")
    plt.xlabel(date_col); plt.ylabel("Pearson r"); plt.tight_layout(); plt.show()
elif groups['humidity'] is not None:
    plot_line(df, date_col, groups['humidity'], title=f"{groups['humidity']} over time", rolling=30)


### 5.3 Precipitation / Cloud / Visibility

In [ ]:

if groups['precip'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['precip'], target_col,
                   title=f"{groups['precip']} vs {target_col}", rolling=7)
    plot_scatter(df, groups['precip'], target_col, title=f"Scatter: {groups['precip']} vs {target_col}")
elif groups['precip'] is not None:
    plot_line(df, date_col, groups['precip'], title=f"{groups['precip']} over time", rolling=7)

if groups['cloudcover'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['cloudcover'], target_col,
                   title=f"{groups['cloudcover']} vs {target_col}", rolling=30)
elif groups['cloudcover'] is not None:
    plot_line(df, date_col, groups['cloudcover'], title=f"{groups['cloudcover']} over time", rolling=30)

if groups['visibility'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['visibility'], target_col,
                   title=f"{groups['visibility']} vs {target_col}", rolling=30)
elif groups['visibility'] is not None:
    plot_line(df, date_col, groups['visibility'], title=f"{groups['visibility']} over time", rolling=30)


### 5.4 Wind

In [ ]:

if groups['windspeed'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['windspeed'], target_col,
                   title=f"{groups['windspeed']} vs {target_col}", rolling=30)
    plot_box_by_quantiles(df, groups['windspeed'], target_col, q=4,
                          title=f"{target_col} by {groups['windspeed']} quantiles")
elif groups['windspeed'] is not None:
    plot_line(df, date_col, groups['windspeed'], title=f"{groups['windspeed']} over time", rolling=30)

if groups['windgust'] is not None:
    plot_line(df, date_col, groups['windgust'], title=f"{groups['windgust']} over time", rolling=30)
    if target_col is not None:
        rc = rolling_corr(df[groups['windgust']], df[target_col], window=90)
        plt.figure(figsize=(12,3)); plt.plot(df[date_col], rc)
        plt.title(f"Rolling (90d) Corr: {groups['windgust']} vs {target_col}")
        plt.xlabel(date_col); plt.ylabel("Pearson r"); plt.tight_layout(); plt.show()


### 5.5 Solar Radiation & Pressure

In [ ]:

if groups['solarradiation'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['solarradiation'], target_col,
                   title=f"{groups['solarradiation']} vs {target_col}", rolling=30)
    rc = rolling_corr(df[groups['solarradiation']], df[target_col], window=90)
    plt.figure(figsize=(12,3)); plt.plot(df[date_col], rc)
    plt.title(f"Rolling (90d) Corr: {groups['solarradiation']} vs {target_col}")
    plt.xlabel(date_col); plt.ylabel("Pearson r"); plt.tight_layout(); plt.show()
elif groups['solarradiation'] is not None:
    plot_line(df, date_col, groups['solarradiation'], title=f"{groups['solarradiation']} over time", rolling=30)

if groups['uvindex'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['uvindex'], target_col,
                   title=f"{groups['uvindex']} vs {target_col}", rolling=30)
elif groups['uvindex'] is not None:
    plot_line(df, date_col, groups['uvindex'], title=f"{groups['uvindex']} over time", rolling=30)

if groups['sealevelpressure'] is not None and target_col is not None:
    plot_dual_axis(df, date_col, groups['sealevelpressure'], target_col,
                   title=f"{groups['sealevelpressure']} vs {target_col}", rolling=30)
elif groups['sealevelpressure'] is not None:
    plot_line(df, date_col, groups['sealevelpressure'], title=f"{groups['sealevelpressure']} over time", rolling=30)


## 6) Global Correlation (if target exists)

In [ ]:

if target_col is not None:
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != target_col]
    corrs = {}
    for c in num_cols:
        s = df[[c, target_col]].dropna()
        if len(s) < 50: 
            continue
        corrs[c] = s[c].corr(s[target_col])
    corr_ser = pd.Series(corrs).dropna().sort_values(key=lambda s: s.abs(), ascending=False)
    if not corr_ser.empty:
        topk = 20
        top = corr_ser.head(topk)
        display(pd.DataFrame({'feature': top.index, 'pearson_r': top.values}))
        M = np.array(top.values).reshape(-1,1)
        plot_heatmap(M, xticks=[target_col], yticks=list(top.index), title='Top-|corr| with target')
    else:
        print("No sufficient numeric columns for correlation.")
else:
    print("No target detected → skip global correlation.")


## 7) Quality / Wrap‑up

In [ ]:

quality_cols = [c for c in df.columns if (c.startswith('q_last24_') or c.startswith('q_win')) and ('hours_' in c)]
if quality_cols:
    for qc in quality_cols[:6]:
        plot_line(df, date_col, qc, title=f'Quality Flag over time: {qc}', rolling=7)
else:
    print("No quality flag columns detected.")


**Kết luận**

- Notebook tự động **không tạo target mới** và chỉ dùng cột có sẵn.
- Nếu phát hiện được target, các phần correlation/relationship sẽ được bật; nếu không, notebook vẫn cung cấp EDA mô tả đầy đủ.
- Bạn có thể đặt thủ công `target_col` ở cell *Target Detection* để cố định một cột cụ thể khi cần.